# ML-03 — Frame Your Lane as an ML Task

This notebook frames the FlyRank content-refresh lane as a **ranking/scoring** problem. The goal is decision-support: identify pages that appear to have an opportunity for review, rather than claiming to predict Google's ranking algorithm.

All data used below is the small anonymized FlyRank teaching dataset. No client names, URLs, titles, or private queries are used.

## 1. My lane as an ML task (type)

**Task type: Scoring / Ranking.**

The practical task is to assign each content item an opportunity score and then rank pages from higher to lower review priority. This is more useful here than producing only a yes/no class because a content team needs an ordered queue of pages to investigate first. The score is decision-support and is not a claim about Google's ranking algorithm.

In [ ]:
task_type = 'scoring / ranking'
reason = 'Produce an ordered review queue so the highest-opportunity pages can be investigated first.'
print(f'Task type: {task_type}')
print(f'Reason: {reason}')

## 2. Target or proxy

**Ideal target:** whether a page is under-capturing clicks relative to its potential given its search position and context.

**Measurable proxy:** `ctr_gap`, defined as the page's observed CTR minus the median CTR of pages in the same `position_tier`. A strongly negative gap means the page is receiving fewer clicks than comparable pages at a similar position.

This is a proxy, not a ground-truth measure of content quality. CTR can also be affected by search intent, SERP features, seasonality, brand demand, and other factors.

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA_URL)

required = {'ctr', 'position_tier', 'content_id', 'client_id'}
missing = required - set(df.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = df['ctr'] - tier_median_ctr

print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Proxy created: ctr_gap')
print('Negative ctr_gap:', int((df['ctr_gap'] < 0).sum()))

## 3. Success metric

**Primary metric: Precision@K, with K = 20 and K = 50.**

For the ranked review queue, Precision@K is the fraction of the top-K pages that satisfy the measurable opportunity definition. A useful ranking should concentrate likely opportunities near the top of the queue rather than spreading them randomly.

For this framing notebook, the measurable opportunity proxy is `ctr_gap < 0`. The final model notebook can replace this simple proxy with the approved label/definition used by the pipeline. The random baseline is the overall opportunity rate.

In [ ]:
def precision_at_k(frame, score_col, k):
    top_k = frame.nlargest(k, score_col)
    return top_k['opportunity_proxy'].mean()

df['opportunity_proxy'] = (df['ctr_gap'] < 0).astype(int)
# Larger score means a higher review priority, so invert the negative gap.
df['opportunity_score'] = -df['ctr_gap']

base_rate = df['opportunity_proxy'].mean()
p20 = precision_at_k(df, 'opportunity_score', 20)
p50 = precision_at_k(df, 'opportunity_score', 50)

print(f'Opportunity base rate: {base_rate:.3f}')
print(f'Precision@20: {p20:.3f}')
print(f'Precision@50: {p50:.3f}')

## 4. The unit of analysis, as a real dataframe

**One row represents one anonymized content item/page.** The starter dataset contains page-level measurements aggregated over a trailing 90-day window. `content_id` identifies the pseudonymous page and `client_id` identifies the pseudonymous client group.

The preview below deliberately shows analytical columns only and does not expose private client information.

In [ ]:
preview_cols = [
    'content_id', 'client_id', 'content_type', 'impressions_90d',
    'clicks_90d', 'ctr', 'avg_position', 'position_tier',
    'days_since_last_update', 'trend_direction'
]
preview = df[preview_cols].head(10)
display(preview)
print('One row = one anonymized content item/page.')
print('Shape:', df.shape)
print('Unique content IDs:', df['content_id'].nunique())

## 5. Why ML beats a fixed rule here

A single rule such as `if CTR < X: review` is too rigid because CTR depends on several interacting factors: search position, impressions, content type, content age, update recency, search intent, and traffic context. The same CTR can mean something different for pages with different positions and volumes.

An ML model can learn combinations of these measurable signals from historical outcomes and produce a ranked queue. This does not make the model causal or perfect: the result remains directional decision-support and should be validated before operational use.

A fixed rule is still useful as a transparent baseline. ML is justified only if it consistently improves the agreed ranking metric on held-out data.

In [ ]:
# Simple sanity checks supporting the framing.
assert df['content_id'].is_unique, 'Expected one row per content item.'
assert df['ctr'].notna().all(), 'CTR should be available for the supplied teaching slice.'
assert df['opportunity_proxy'].isin([0, 1]).all()
print('Sanity checks passed.')
print('Candidate input signals include position, traffic volume, content type, age, and update recency.')

## Self-check

- [x] Every section is filled with markdown reasoning and supporting code.
- [x] The notebook is designed to run top to bottom without requiring private credentials.
- [x] No client names, URLs, titles, or private queries are used.
- [x] Claims use careful terms such as observed, measured, directional, proxy, and decision-support.
- [x] The notebook belongs under `work/notebooks/` and is ready to commit to the submission repository.

**Submission note:** run all cells in Colab/Jupyter once, confirm there are no errors, save the executed notebook, and submit the repository URL on the assignment card.